# Entrenamiento, artefacto y evaluación congelada
Juan Camilo Gonzales y Juan Jose Solarte. Este notebook importa el entrenamiento y las métricas del servicio. TF-IDF se ajusta dentro de Pipeline solo con train; C se elige solo con validation. El umbral 0,45 sigue provisional: H-01=0,70 no se reemplaza como decisión aprobada.

El test ya fue evaluado en el proyecto. Aquí se reproduce el resultado del artefacto congelado, no una nueva prueba ciega ni un ciclo de optimización. No se vuelve a ajustar el modelo después de observar test.

In [ ]:
from pathlib import Path
import sys, os, json, tempfile
root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'ai_service' / 'train.py').exists())
if str(root) not in sys.path: sys.path.insert(0, str(root))
from ai_service.train import train, LABELS
from ai_service.promote import promote
from ai_service.evaluate_artifact import evaluate_frozen
output = Path(os.environ.get('NLP_NOTEBOOK_OUTPUT', tempfile.mkdtemp(prefix='nlp-academico-')))
dataset = root / 'ai_service/data/correos_sinteticos_v0.3.jsonl'
model, validation_file = train(dataset, output / 'candidate', False)
manifest = promote(model, output / 'frozen')
manifest

In [ ]:
import pandas as pd
validation = evaluate_frozen(output / 'frozen/model.joblib', dataset, include_test=False)
pd.DataFrame(validation['validationThresholds'])

## Congelar antes del test
La comparación anterior describe sensibilidad en validation; no selecciona un umbral nuevo. Se mantiene 0,45 por configuración del MVP, sin inferir que sea óptimo. No editar hiperparámetros ni el umbral a partir de las celdas siguientes.

In [ ]:
report = evaluate_frozen(output / 'frozen/model.joblib', dataset, include_test=True)
(output / 'notebook-evaluation.json').write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')
pd.DataFrame({name: {'Macro-F1': values['macroF1'], 'accuracy': values['accuracy']} for name, values in report['evaluations']['test'].items() if name != 'routingSimulation'}).T

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import ConfusionMatrixDisplay
metrics = report['evaluations']['test']['classifier']
display = ConfusionMatrixDisplay(np.asarray(metrics['confusionMatrix']), display_labels=LABELS).plot(xticks_rotation=35)
plt.title('Test sintético: artefacto congelado')
plt.tight_layout()
display.figure_.savefig(output / 'matriz_confusion_test.png', dpi=150, bbox_inches='tight')
plt.show()
pd.DataFrame(metrics['perClass']).T[['precision', 'recall', 'f1-score', 'support']]

In [ ]:
print('Enrutamiento simulado:', report['evaluations']['test']['routingSimulation'])
pd.DataFrame(report['errors']['test'])

## Interpretación
23/24 o 20/20 no permiten prometer esa exactitud en producción. La cobertura automática es simulada con probabilidades no calibradas, no una medición de correcciones humanas. Ampliar el corpus exige un nuevo protocolo de evaluación sin reutilizar este test para seleccionar. Jupyter no se usa para servir solicitudes HTTP.